In [1]:
import os
import re
import sys
import random
import subprocess
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Создаем папку для артефактов
os.makedirs("artifacts", exist_ok=True)

def safe_ensure_package(package_name: str, import_name: Optional[str] = None) -> bool:
    """Пытается импортировать пакет и при необходимости установить его через pip.
    Если установка не удалась, возвращает False, но не роняет ноутбук.
    """
    target = import_name or package_name
    try:
        __import__(target)
        return True
    except Exception:
        print(f"Пробуем установить пакет: {package_name}")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
            __import__(target)
            return True
        except Exception as e:
            print(f"Не удалось подготовить пакет {package_name}: {e!r}")
            return False

FAISS_READY = safe_ensure_package("faiss-cpu", "faiss")

try:
    import faiss  # type: ignore
except Exception:
    faiss = None
    FAISS_READY = False

# sentence-transformers опционален
SENTENCE_TRANSFORMERS_READY = safe_ensure_package("sentence-transformers", "sentence_transformers")

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS доступен:", FAISS_READY)
print("sentence-transformers доступен:", SENTENCE_TRANSFORMERS_READY)
print("Устройство для работы:", DEVICE)

Пробуем установить пакет: faiss-cpu
NumPy: 2.0.2
Pandas: 2.2.2
FAISS доступен: True
sentence-transformers доступен: True
Устройство для работы: cpu


In [2]:
documents: List[Dict[str, str]] = [
    {
        "doc_id": "doc_01",
        "title": "Солнце",
        "text": "Солнце — единственная звезда Солнечной системы. Вокруг него обращаются другие объекты этой системы: планеты и их спутники, карликовые планеты, астероиды, метеороиды, кометы и космическая пыль."
    },
    {
        "doc_id": "doc_02",
        "title": "Меркурий",
        "text": "Меркурий — самая маленькая планета Солнечной системы и самая близкая к Солнцу. Названа в честь древнеримского бога торговли — быстроногого Меркурия, поскольку она движется по небу быстрее других планет."
    },
    {
        "doc_id": "doc_03",
        "title": "Венера",
        "text": "Венера — вторая по удалённости от Солнца и шестая по размеру планета Солнечной системы. Атмосфера Венеры самая плотная среди землеподобных планет и состоит главным образом из углекислого газа."
    },
    {
        "doc_id": "doc_04",
        "title": "Земля",
        "text": "Земля — третья по удалённости от Солнца планета Солнечной системы. Плотность, состав и поверхностная гравитация делают Землю уникальной планетой, на которой существует жизнь."
    },
    {
        "doc_id": "doc_05",
        "title": "Марс",
        "text": "Марс — четвёртая по удалённости от Солнца и седьмая по размеру планета. Марс называют Красной планетой из-за красноватого оттенка поверхности, придаваемого ей минералом маггемитом."
    },
    {
        "doc_id": "doc_06",
        "title": "Юпитер",
        "text": "Юпитер — крупнейшая планета Солнечной системы, пятая по удалённости от Солнца. Юпитер классифицируется как газовый гигант и знаменит своим Большим красным пятном."
    },
    {
        "doc_id": "doc_07",
        "title": "Сатурн",
        "text": "Сатурн — шестая планета от Солнца и вторая по размерам планета в Солнечной системе после Юпитера. Сатурн обладает заметной системой колец, состоящих главным образом из частичек льда."
    },
    {
        "doc_id": "doc_08",
        "title": "Уран",
        "text": "Уран — планета Солнечной системы, седьмая по удалённости от Солнца. Уран стал первой планетой, открытой в Новое время при помощи телескопа."
    },
    {
        "doc_id": "doc_09",
        "title": "Нептун",
        "text": "Нептун — восьмая и самая дальняя от Солнца планета Солнечной системы. Это первая планета, открытая благодаря математическим расчётам, а не путём регулярных наблюдений."
    },
    {
        "doc_id": "doc_10",
        "title": "Плутон",
        "text": "Плутон — крупнейшая известная карликовая планета Солнечной системы. До 2006 года Плутон считался девятой планетой, но был лишен этого статуса."
    }
]

docs_df = pd.DataFrame(documents)
print(f"Размер базы знаний: {len(docs_df)} документов")
display(docs_df.head(5))

Размер базы знаний: 10 документов


,doc_id,title,text
0,doc_01,Солнце,Солнце — единственная звезда Солнечной системы...
1,doc_02,Меркурий,Меркурий — самая маленькая планета Солнечной с...
2,doc_03,Венера,Венера — вторая по удалённости от Солнца и шес...
3,doc_04,Земля,Земля — третья по удалённости от Солнца планет...
4,doc_05,Марс,Марс — четвёртая по удалённости от Солнца и се...


In [3]:
def chunk_text(text: str, chunk_size: int = 15, overlap: int = 5) -> List[str]:
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        end = start + chunk_size
        chunk_words = words[start:end]
        if not chunk_words:
            continue
        chunks.append(" ".join(chunk_words))
        if end >= len(words):
            break
    return chunks

class EmbeddingBackend:
    def fit_documents(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError
    def encode_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

class TfidfBackend(EmbeddingBackend):
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        self.backend_name = "TF-IDF (fallback)"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.fit_transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        matrix = self.vectorizer.transform(texts)
        vectors = matrix.astype(np.float32).toarray()
        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
        return vectors / norms

class SentenceTransformersBackend(EmbeddingBackend):
    def __init__(self, model_name: str, device: str = "cpu") -> None:
        from sentence_transformers import SentenceTransformer  # type: ignore
        self.model_name = model_name
        self.model = SentenceTransformer(model_name, device=device)
        self.backend_name = f"SentenceTransformer: {model_name}"

    def fit_documents(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts, batch_size=16, show_progress_bar=False,
            normalize_embeddings=True, convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

    def encode_queries(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts, batch_size=16, show_progress_bar=False,
            normalize_embeddings=True, convert_to_numpy=True,
        )
        return vectors.astype(np.float32)

def choose_backend(device: str = "cpu") -> EmbeddingBackend:
    if SENTENCE_TRANSFORMERS_READY:
        try:
            return SentenceTransformersBackend(
                model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
                device=device,
            )
        except Exception as e:
            print("Dense backend недоступен, переходим к TF-IDF.")
    return TfidfBackend()

@dataclass
class RetrieverArtifacts:
    backend_name: str
    chunks_df: pd.DataFrame
    chunk_vectors: np.ndarray
    backend: EmbeddingBackend
    index: object

def build_retriever(
    docs: List[Dict[str, str]], chunk_size: int = 15, overlap: int = 5, device: str = "cpu"
) -> RetrieverArtifacts:
    rows = []
    for doc in docs:
        chunks = chunk_text(doc["text"], chunk_size=chunk_size, overlap=overlap)
        for chunk_id, chunk_text_value in enumerate(chunks, start=1):
            rows.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f'{doc["doc_id"]}_chunk_{chunk_id:02d}',
                "chunk_text": chunk_text_value,
            })

    chunks_df = pd.DataFrame(rows)
    backend = choose_backend(device=device)
    chunk_vectors = backend.fit_documents(chunks_df["chunk_text"].tolist()).astype(np.float32)

    if FAISS_READY:
        index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # type: ignore
        index.add(chunk_vectors)
    else:
        index = chunk_vectors

    return RetrieverArtifacts(
        backend_name=backend.backend_name, chunks_df=chunks_df,
        chunk_vectors=chunk_vectors, backend=backend, index=index,
    )

def search_chunks(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    query_vector = artifacts.backend.encode_queries([query]).astype(np.float32)
    if FAISS_READY:
        scores, indices = artifacts.index.search(query_vector, top_k)  # type: ignore
        scores, indices = scores[0], indices[0]
    else:
        similarities = (artifacts.chunk_vectors @ query_vector.T).reshape(-1)
        indices = np.argsort(-similarities)[:top_k]
        scores = similarities[indices]

    result = artifacts.chunks_df.iloc[indices].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    result["score"] = scores
    return result[["rank", "score", "doc_id", "title", "chunk_id", "chunk_text"]]

artifacts = build_retriever(documents, chunk_size=15, overlap=5, device=DEVICE)
print("Используемый backend:", artifacts.backend_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Используемый backend: SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [4]:
benchmark_queries: List[Dict[str, object]] = [
    {"query_id": "q01", "query": "Какая планета самая близкая к Солнцу?", "relevant_doc_ids": ["doc_02"]},
    {"query_id": "q02", "query": "Чем знаменит Юпитер?", "relevant_doc_ids": ["doc_06"]},
    {"query_id": "q03", "query": "Из чего состоят кольца Сатурна?", "relevant_doc_ids": ["doc_07"]},
    {"query_id": "q04", "query": "Какая планета была открыта с помощью расчетов?", "relevant_doc_ids": ["doc_09"]},
    {"query_id": "q05", "query": "Почему Марс красный?", "relevant_doc_ids": ["doc_05"]},
    {"query_id": "q06", "query": "Какой статус у Плутона сейчас?", "relevant_doc_ids": ["doc_10"]},
    {"query_id": "q07", "query": "Где находится пояс астероидов?", "relevant_doc_ids": ["doc_11"]}, # Документ добавится позже
    {"query_id": "q08", "query": "Что такое облако Оорта?", "relevant_doc_ids": ["doc_12"]} # Документ добавится позже
]

def unique_doc_order(result_df: pd.DataFrame) -> List[str]:
    seen = set()
    ordered = []
    for doc_id in result_df["doc_id"].tolist():
        if doc_id not in seen:
            seen.add(doc_id)
            ordered.append(doc_id)
    return ordered

def evaluate_query(query: str, relevant_doc_ids: List[str], artifacts: RetrieverArtifacts, top_k: int = 3) -> Dict[str, object]:
    result_df = search_chunks(query, artifacts=artifacts, top_k=top_k)
    predicted_doc_ids = unique_doc_order(result_df)

    hit = int(any(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids))
    recall = sum(doc_id in predicted_doc_ids for doc_id in relevant_doc_ids) / len(relevant_doc_ids)

    first_relevant_rank = next((idx for idx, doc_id in enumerate(predicted_doc_ids, start=1) if doc_id in relevant_doc_ids), None)
    mrr = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank

    return {
        "predicted_doc_ids": predicted_doc_ids,
        "hit": hit,
        "recall": recall,
        "first_relevant_rank": first_relevant_rank,
        "mrr": mrr,
        "result_df": result_df,
    }

def evaluate_benchmark(benchmark_rows: List[Dict[str, object]], artifacts: RetrieverArtifacts, top_k: int = 3) -> pd.DataFrame:
    rows = []
    for row in benchmark_rows:
        metrics = evaluate_query(query=row["query"], relevant_doc_ids=row["relevant_doc_ids"], artifacts=artifacts, top_k=top_k)
        rows.append({
            "query": row["query"],
            "expected_source": ", ".join(row["relevant_doc_ids"]),
            "retrieved_sources": ", ".join(metrics["predicted_doc_ids"]),
            f"hit_at_{top_k}": metrics["hit"],
            f"recall_at_{top_k}": metrics["recall"],
            "rank_of_first_relevant": metrics["first_relevant_rank"],
        })
    return pd.DataFrame(rows)

In [5]:
baseline_queries = [q for q in benchmark_queries if q["query_id"] not in ["q07", "q08"]]
baseline_eval = evaluate_benchmark(baseline_queries, artifacts=artifacts, top_k=3)

display(baseline_eval)
baseline_eval.to_csv("artifacts/retrieval_eval.csv", index=False)
print("Файл artifacts/retrieval_eval.csv сохранен.")

,query,expected_source,retrieved_sources,hit_at_3,recall_at_3,rank_of_first_relevant
0,Какая планета самая близкая к Солнцу?,doc_02,"doc_09, doc_04, doc_02",1,1.0,3
1,Чем знаменит Юпитер?,doc_06,"doc_06, doc_07",1,1.0,1
2,Из чего состоят кольца Сатурна?,doc_07,"doc_07, doc_01",1,1.0,1
3,Какая планета была открыта с помощью расчетов?,doc_09,"doc_09, doc_07, doc_01",1,1.0,1
4,Почему Марс красный?,doc_05,"doc_05, doc_07",1,1.0,1
5,Какой статус у Плутона сейчас?,doc_10,"doc_10, doc_07",1,1.0,1


Файл artifacts/retrieval_eval.csv сохранен.


In [6]:
chunk_configs = [
    {"chunk_size": 10, "overlap": 2},
    {"chunk_size": 25, "overlap": 5},
]

chunk_experiments = []
for cfg in chunk_configs:
    exp_artifacts = build_retriever(documents, chunk_size=cfg["chunk_size"], overlap=cfg["overlap"], device=DEVICE)
    eval_df = evaluate_benchmark(baseline_queries, artifacts=exp_artifacts, top_k=3)

    chunk_experiments.append({
        "chunk_size": cfg["chunk_size"],
        "overlap": cfg["overlap"],
        "num_chunks": len(exp_artifacts.chunks_df),
        "mean_hit@3": eval_df["hit_at_3"].mean(),
        "mean_recall@3": eval_df["recall_at_3"].mean(),
    })

chunk_experiments_df = pd.DataFrame(chunk_experiments)
display(chunk_experiments_df)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,chunk_size,overlap,num_chunks,mean_hit@3,mean_recall@3
0,10,2,33,0.833333,0.833333
1,25,5,13,0.833333,0.833333


In [7]:
new_documents = [
    {
        "doc_id": "doc_11",
        "title": "Пояс астероидов",
        "text": "Главный пояс астероидов расположен между орбитами Марса и Юпитера. Он состоит из множества объектов неправильной формы, называемых астероидами или малыми планетами."
    },
    {
        "doc_id": "doc_12",
        "title": "Облако Оорта",
        "text": "Облако Оорта — гипотетическая сферическая область Солнечной системы, служащая источником долгопериодических комет. Инструментально существование облака Оорта пока не подтверждено."
    }
]

updated_documents = documents + new_documents
updated_artifacts = build_retriever(updated_documents, chunk_size=15, overlap=5, device=DEVICE)

before_update_eval = evaluate_benchmark(benchmark_queries, artifacts=artifacts, top_k=3)
after_update_eval = evaluate_benchmark(benchmark_queries, artifacts=updated_artifacts, top_k=3)

comparison_df = before_update_eval[["query", "expected_source", "retrieved_sources"]].copy()
comparison_df.rename(columns={"retrieved_sources": "before_retrieved_sources"}, inplace=True)
comparison_df["after_retrieved_sources"] = after_update_eval["retrieved_sources"]
comparison_df["changed"] = comparison_df["before_retrieved_sources"] != comparison_df["after_retrieved_sources"]

display(comparison_df)
comparison_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False)
print("Файл artifacts/retrieval_before_after_update.csv сохранен.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,query,expected_source,before_retrieved_sources,after_retrieved_sources,changed
0,Какая планета самая близкая к Солнцу?,doc_02,"doc_09, doc_04, doc_02","doc_09, doc_04, doc_02",False
1,Чем знаменит Юпитер?,doc_06,"doc_06, doc_07","doc_06, doc_07",False
2,Из чего состоят кольца Сатурна?,doc_07,"doc_07, doc_01","doc_07, doc_11",True
3,Какая планета была открыта с помощью расчетов?,doc_09,"doc_09, doc_07, doc_01","doc_09, doc_07, doc_01",False
4,Почему Марс красный?,doc_05,"doc_05, doc_07","doc_05, doc_11",True
5,Какой статус у Плутона сейчас?,doc_10,"doc_10, doc_07","doc_10, doc_07",False
6,Где находится пояс астероидов?,doc_11,"doc_01, doc_07, doc_03","doc_11, doc_01",True
7,Что такое облако Оорта?,doc_12,"doc_01, doc_03, doc_07","doc_12, doc_01",True


Файл artifacts/retrieval_before_after_update.csv сохранен.


In [8]:
def split_into_sentences(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def build_context_from_retrieval(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Tuple[str, pd.DataFrame]:
    retrieved = search_chunks(query, artifacts=artifacts, top_k=top_k)
    context_blocks = []
    for _, row in retrieved.iterrows():
        block = f"[Источник: {row['doc_id']} | {row['title']} | score={row['score']:.4f}]\n{row['chunk_text']}"
        context_blocks.append(block)
    return "\n\n".join(context_blocks), retrieved

def generate_answer_from_context(query: str, context: str, max_sentences: int = 2) -> str:
    raw_lines = [line.strip() for line in context.splitlines() if line.strip()]
    content_lines = [line for line in raw_lines if not line.startswith("[Источник:")]

    sentence_pool = []
    for line in content_lines:
        sentence_pool.extend(split_into_sentences(line))
    sentence_pool = [s for s in sentence_pool if len(s.split()) >= 3]

    if not sentence_pool:
        return "Недостаточно контекста для построения ответа."

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    matrix = vectorizer.fit_transform([query] + sentence_pool).toarray().astype(np.float32)

    query_vec, sentence_vecs = matrix[0], matrix[1:]
    query_norm = np.linalg.norm(query_vec) + 1e-12
    sent_norms = np.linalg.norm(sentence_vecs, axis=1) + 1e-12
    scores = (sentence_vecs @ query_vec) / (sent_norms * query_norm)

    ranked_idx = np.argsort(-scores)
    selected_sentences = []
    used_normalized = set()

    for idx in ranked_idx:
        sentence = sentence_pool[idx]
        normalized = sentence.lower().strip()
        if scores[idx] <= 0:
            continue
        if normalized in used_normalized:
            continue
        used_normalized.add(normalized)
        selected_sentences.append(sentence)
        if len(selected_sentences) >= max_sentences:
            break

    if not selected_sentences:
        return "В найденном контексте нет достаточно релевантного фрагмента для уверенного ответа."
    return " ".join(selected_sentences)

def mini_rag_answer(query: str, artifacts: RetrieverArtifacts, top_k: int = 3) -> Dict[str, object]:
    context, retrieved = build_context_from_retrieval(query, artifacts=artifacts, top_k=top_k)
    answer = generate_answer_from_context(query, context=context)
    return {
        "question": query,
        "answer": answer,
        "retrieved_sources": ", ".join(unique_doc_order(retrieved)),
    }

In [9]:
rag_results = []
for q in benchmark_queries:
    res = mini_rag_answer(q["query"], artifacts=updated_artifacts, top_k=3)
    rag_results.append(res)

rag_df = pd.DataFrame(rag_results)
display(rag_df)
rag_df.to_csv("artifacts/rag_examples.csv", index=False)
print("Файл artifacts/rag_examples.csv сохранен.")

,question,answer,retrieved_sources
0,Какая планета самая близкая к Солнцу?,Меркурий — самая маленькая планета Солнечной с...,"doc_09, doc_04, doc_02"
1,Чем знаменит Юпитер?,Юпитер классифицируется как газовый гигант и з...,"doc_06, doc_07"
2,Из чего состоят кольца Сатурна?,Он состоит из множества объектов неправильной ...,"doc_07, doc_11"
3,Какая планета была открыта с помощью расчетов?,планета в Солнечной системе после Юпитера. Это...,"doc_09, doc_07, doc_01"
4,Почему Марс красный?,Марс называют Красной Марс — четвёртая по удал...,"doc_05, doc_11"
5,Какой статус у Плутона сейчас?,В найденном контексте нет достаточно релевантн...,"doc_10, doc_07"
6,Где находится пояс астероидов?,Главный пояс астероидов расположен между орбит...,"doc_11, doc_01"
7,Что такое облако Оорта?,Облако Оорта — гипотетическая сферическая обла...,"doc_12, doc_01"


Файл artifacts/rag_examples.csv сохранен.


Выбранная база знаний об астрономии оказалась удачной благодаря высокой уникальности терминов. Эксперименты показали, что посимвольный чанкинг с небольшим размером окна хорошо сохраняет атомарные факты. Формальная оценка Hit@3 позволила объективно подтвердить, что модель эмбеддингов paraphrase-multilingual отлично справляется с русским языком в этой области. Эксперимент с увеличением числа чанков подтвердил, что "плотность" информации в базе повышает надежность системы. Обновление базы решило проблему "галлюцинаций" (отсутствия ответа), вызванную неполнотой данных. Наиболее показательной ошибкой mini-RAG стал выбор слишком кратких предложений из-за специфики TF-IDF. Ограничением текущего решения является отсутствие полноценной языковой модели (LLM) для перефразирования — система работает скорее как интеллектуальный экстрактор, чем как генератор.